## Download the HMDA Data for KS
Note: I had some issues downloading the files so this is a bit over-engineered for a download. 

In [0]:
# imports
import os
import time
import requests
import glob
import pandas as pd

In [0]:
OUTPUT_DIR = "./data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

downloads = [
    {"url": "https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?states=KS&years=2018&loan_purposes=1", "filename": "hmda_ks_2018.csv"},
    {"url": "https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?states=KS&years=2019&loan_purposes=1", "filename": "hmda_ks_2019.csv"},
    {"url": "https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?states=KS&years=2020&loan_purposes=1", "filename": "hmda_ks_2020.csv"},
    {"url": "https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?states=KS&years=2021&loan_purposes=1", "filename": "hmda_ks_2021.csv"},
    {"url": "https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?states=KS&years=2022&loan_purposes=1", "filename": "hmda_ks_2022.csv"},
    {"url": "https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?states=KS&years=2023&loan_purposes=1", "filename": "hmda_ks_2023.csv"},
    {"url": "https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?states=KS&years=2024&loan_purposes=1", "filename": "hmda_ks_2024.csv"},
    {"url": "https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?states=MO&years=2025&loan_purposes=1", "filename": "hmda_mo_2025.csv"}
]

MAX_RETRIES = 3

for item in downloads:
    url, filename = item["url"], item["filename"]
    output_path = os.path.join(OUTPUT_DIR, filename)

    # Skip download if the file already exists
    if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
        print(f"Skipping {filename} (already exists, {os.path.getsize(output_path):,} bytes)")
        continue

    # Retryable HTTP status codes (server-side / transient errors)
    RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(url, stream=True, timeout=60)
            response.raise_for_status()
            break
        except requests.exceptions.HTTPError as e:
            status_code = e.response.status_code if e.response is not None else None
            if status_code in RETRYABLE_STATUS_CODES and attempt < MAX_RETRIES:
                wait = 5 * attempt
                print(f"[{filename}] HTTP {status_code} (attempt {attempt}/{MAX_RETRIES}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            if attempt < MAX_RETRIES:
                wait = 5 * attempt
                print(f"[{filename}] {type(e).__name__}: {e} (attempt {attempt}/{MAX_RETRIES}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

    with open(output_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

    print(f"Downloaded {filename} -> {output_path} ({os.path.getsize(output_path):,} bytes)")

In [0]:
csv_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.csv")))
print(f"Found {len(csv_files)} CSV file(s):")
for f in csv_files:
    print(f"  {os.path.basename(f)} ({os.path.getsize(f):,} bytes)")

df = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)
print(f"\nCombined DataFrame shape: {df.shape}")
display(df.head())

In [0]:
df.info()

In [0]:
df.describe()